# 10. Molecular Dynamics

We're now ready to take on our most complicated project so far - modeling multiple interacting particles. We'll start simple, with only **two particles**, and then try to extend it to **any number of particles**.

## Two particles
Let's begin with two particles. First thing is to set up the essential parameters. We'll model inter-particle interactions using the LJ potential that we've studied multiple times this semester. 

For clarity we'll be using SI units throughout.

Choosing dt deserves special attention here. We mentioned earlier that simulating atomic and molecular motion require timesteps on the order of femtoseconds. Can we actually derive this based on the LJ potential parameters we have? In the following we'll using `SymPy` to calculate the *analytical* solution to help us determine the `dt` setup in our *numerical* calculations. 

(There are many ways to get the 2nd derivative. For alternative methods, see "Harmonic approx.pdf" on Canvas, where we can find an effective spring constant of $k=(4\epsilon) 18/2^{1/3}\sigma^2$ used in our code.

In [1]:
#The following code was adapted based on code generated from Claude & ChatGPT
import sympy as sp

# Symbols
r, epsilon, sigma = sp.symbols('r epsilon sigma', positive=True)

# Lennard-Jones potential
V = 4 * epsilon * ((sigma/r)**12 - (sigma/r)**6)

# First and second derivatives
dV_dr = sp.diff(V, r)
d2V_dr2 = sp.diff(V, r, 2)

print("V(r) =")
sp.pprint(V)

print("\ndV/dr =")
sp.pprint(sp.simplify(dV_dr))

print("\nd²V/dr² =")
sp.pprint(sp.simplify(d2V_dr2))

r0 = 2**(sp.Rational(1, 6)) * sigma
k = sp.simplify(d2V_dr2.subs(r, r0))

print("d²V/dr² at equilibrium:")
sp.pprint(k)

V(r) =
    ⎛   6    12⎞
    ⎜  σ    σ  ⎟
4⋅ε⋅⎜- ── + ───⎟
    ⎜   6    12⎟
    ⎝  r    r  ⎠

dV/dr =
      6 ⎛ 6      6⎞
24⋅ε⋅σ ⋅⎝r  - 2⋅σ ⎠
───────────────────
         13        
        r          

d²V/dr² =
      6 ⎛     6       6⎞
24⋅ε⋅σ ⋅⎝- 7⋅r  + 26⋅σ ⎠
────────────────────────
           14           
          r             
d²V/dr² at equilibrium:
    2/3  
36⋅2   ⋅ε
─────────
    2    
   σ     


In [2]:
import matplotlib.pyplot as plt
import numpy as np

def move_RK2(diffeq, y, dt):
    ydot = diffeq( y )
    y_half = y + 1/2 * ydot * dt
    ydot_half  =  diffeq( y_half)
    y = y + ydot_half * dt
    return y
    

def move_Euler(diffeq, y, dt):
    ydot = diffeq( y )
    y = y+ ydot * dt
    return y

epsilon= 4e-4 * 27.211 * 1.6e-19 # in J
bohr = 0.53e-10 
sigma=2.4 * bohr # in m
mass= 40 * 1e-3/6.02e23 # in kg

nparticles = 2 # Total of 2 particles
R = 10 * bohr # Our simulation box is a square that spans -R to +R in both the x and y directions.

k = 4*epsilon * 18/2**(1/3)/sigma**2
omega = np.sqrt(k/mass)

period = 2*np.pi/omega
dt = 0.01 * period

After these parameter setup, we'll need the basic LJ forces we derived in homework earlier.

In [4]:
def flj(x):    
    return -4*epsilon*(-12*sigma**12/x**13 \
                       +6*sigma**6/x**7)

In the following we'll break up into four teams to setup the four essential blocks of code we need.
- Team 1: a function for calculting forces given the coordinates of particles
- Team 2: a standard `diffeq()` function 
- Team 3: an ODE solver section, including setting up proper boundary conditions
- Team 4: a plotting and animation section

A template is given below to get started.

The four tasks are inter-dependent! To make sure one team's output gets fed into another team's input following the same data structure, we'll use the following conventions:
- "positions" will be a 2D array with dimensions = `nparticles` $\times$ `2`
- "forces" will also be a 2D array with dimensions = `nparticles ` $\times$ ` 2`
- `y` will be a 1D array with size = `4 * nparticles`, `[x1,z1,x2,z2,....vx1,vz1,vx2,vz2....]`
- `y_list` will be a 2D array with dimensions `n_timesteps ` $\times$ ` 4 * nparticles` 


In [ ]:
# Team 1

def calc_forces(position):

    x1 = 
    z1 = 
    x2 = 
    z2 = 
    
    r12 = 
    fx1 = 
    fz1 = 
    fx2 = 
    fz2 = 
    forces = np.array([[fx1,fz1],[fx2,fz2]])
    return forces

In [ ]:
# Team 2

def diffeq(y):
    position = 
    a = calc_forces( position) / mass
    ydot = 
    ydot = np.array(ydot)
    return ydot

In [ ]:
# Team 3

y = np.zeros(8)
y[0:4] = np.array([-1,0,1,1])*bohr

t_total = 1000*dt
t_range = np.arange(0, t_total, dt)
y_list = []
    
for t in t_range:
    y = move_RK2( diffeq, y, dt)
    # boundary conditions here          
    y_list.append(y)
    
y_list = np.array(y_list)    
    

In [ ]:
# Team 4

import matplotlib.pyplot as plt
import matplotlib.animation as anim
from IPython.display import HTML

fig, ax = plt.subplots(1)
ax.set_aspect('equal', 'box')
ax.set_xlim(??)
ax.set_ylim(??)
a, =  ax.plot([],[], 'go')


import matplotlib.animation as anim
def animate(i):
    a.set_data(??)
    return a,

frame_skip = 10

ani = anim.FuncAnimation(fig, animate, \
                frames=range(0, len(t_range), frame_skip),\
                interval=10, repeat=True,\
                    blit=True)

plt.close(fig)
HTML(ani.to_jshtml())

.

.

.

(In class discussions)  
The finished version we got back from each team looks like the following.

## Any number of particles

If we can run the simulation for two particles then it's straightforward to generalize it to any number of particles. We just need to consider each pair and add up all the forces.

(Another round of group work to update each block...)

With input from every team, our finished product looks like the following.

Additional discussions:
- initializing a grid of particles
- Speeding up in calc_forces() by calculating flj only once
- Freezing into clusters and a triangular lattice

